# GISLR — Landmark Motion-Over-Time Analysis (v2, GISLR_Stratified)

Diagnostic notebook (pre-training, **not** part of the train pipeline). Measures how
much each landmark moves over time — per-landmark **RMS speed, xy only** — and how
much each of 28 skeletal joint angles changes over time — per-joint **RMS angular
speed** — at three scopes:

| scope | sample | resumable unit | output (under `data/cache/gislr/motion_analysis/`) |
|---|---|---|---|
| 1 — per-video | 50 random videos (seed 42) | video | `per_video/{landmarks,angles}/summary.parquet` + one PNG pair per video |
| 2 — per-category | 10 random signs (seed 42) | sign | `per_category/{landmarks,angles}/summary.parquet` |
| 3 — global | every video in `train.csv`+`test.csv` | 500-video chunk | `global/{landmarks,angles}/summary.parquet` |

**Rebuild, not a re-derivation (TODO §1.8, §7.7).** The original version of this
notebook (`git log` ≤ `1cc5c3c`) read raw `asl-signs` parquet through DuckDB; that
dataset stopped being downloadable 2026-09-16 when GISLR moved to the
self-produced **GISLR_Stratified** npz dataset (`sb.core.paths.gislr_dir()`), which
broke the old notebook (deleted 2026-09-19, findings preserved in
[`docs/reports/motion-energy.md`](../../docs/reports/motion-energy.md)). This
version reads npz directly — no DuckDB needed, since there's no 94K-file JOIN to
push into SQL when metadata already lives in two CSVs — and folds in three
follow-ups the old report flagged as future work rather than re-running the old
formula unchanged:

**Design decisions (vs the original TODO §1 spec and the pre-npz notebook):**
- **z is dropped before the motion-energy computation, not after.** The original
  notebook computed RMS speed on raw xyz and only checked an xy-vs-xyz decomposition
  afterward, on a 50-video sample (report §2.3) — finding **~92% of pose "motion"
  was z-axis depth jitter**, xyz misleadingly ranking pose-head/legs as movers when
  they're image-plane static. Here every scope computes `rms_speed` directly on xy;
  `sb.recognize.interp.motion_energy.landmark_motion_energy` takes whatever
  coordinate subspace it's given, so there's no second xyz pass to reconcile against
  — this **is** the "global xy-only re-run" TODO §1.8 asked for, at all three
  scopes, not just the 50-video sample.
- **Savitzky-Golay smooths the position series *before* the frame-to-frame
  difference**, same window/polyorder as before (7, 2) — a landmark that quivers in
  place smooths flat and scores near-zero, rather than the raw per-frame jitter
  magnitude counting as motion. This is the literal answer to "don't count jitter as
  motion": the smoothing removes the high-frequency residual *before* the derivative
  ever sees it, not as a post-hoc filter on the RMS output.
- **New: 28 joint angles + their rate of change.** Reuses
  `sb.recognize.interp.kinematics.compute_angles` (shoulder/elbow/wrist flex + 10
  finger joints per hand + palm-facing — the same angle geometry the
  feature-discriminability notebook already scored for class-discriminability) and
  adds `joint_angle_motion`'s RMS angular speed (deg/frame) as the angle analogue of
  per-landmark RMS speed: not "where is the joint", but "how fast is it bending".
  **Angles are computed in full xyz**, not xy — unlike a single landmark's raw
  image-plane position, a joint's bend angle is inherently a 3D quantity, and
  collapsing it to 2D would remove exactly the flexion being measured. This is a
  deliberate asymmetry with the xy-only landmark motion energy above, not an
  oversight.
- **Normalized coordinates (TODO §7.7).** Joint angles are computed from
  `center_and_scale`d positions (mid-shoulder-centered, inter-shoulder-scaled) —
  angles are already scale/translation-invariant, but this keeps them on the exact
  position series `kinematics.video_descriptors` uses, so values are directly
  comparable across notebooks. Per-landmark motion energy is deliberately left
  **un-normalized** (raw xy): normalizing would rescale every video by its own
  signer's apparent size, which is a legitimate question for a *discriminability*
  instrument (already covered by `gislr.0.dataset.feature-discriminability.ipynb`)
  but would corrupt *this* notebook's question — "how much does each landmark
  physically move" — by conflating a signer standing closer to the camera with a
  signer moving more.

**Resumability:** every scope runs through the same manifest-driven loop
(`process_units`), unchanged from the pre-npz notebook. Interrupt at any point and
re-run top to bottom — `done` units are skipped, `failed` units are retried.
Per-unit artifacts are written *before* being marked `done`. Each unit now produces
**two** artifacts (landmarks + angles); a unit counts `done` only once both exist.

**NaN policy (landmarks; unchanged):** a landmark's speed at transition t→t+1 counts
toward its RMS only when the landmark was detected at **both** t and t+1 in the raw
data. Absent stretches are interpolated for smoothing continuity but never scored —
so a hand seen in 20% of frames is measured on those frames, not diluted by
interpolated stillness. Landmarks with zero valid transitions get `rms_speed = NaN`.
Angles have no per-landmark detection concept — interpolation has already filled
every gap before an angle is computed, so no additional masking applies there.


In [ ]:
# ============================================================
# Imports & config
# ============================================================
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, gislr_dir
from sb.recognize.interp import motion_energy as me
from sb.recognize.interp.kinematics import ANGLE_NAMES

DATA_DIR = gislr_dir()
MOTION_CACHE_DIR = CACHE_DIR / "gislr" / "motion_analysis"
MOTION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42  # every sampling step below uses this seed
SAVGOL_WINDOW = 7  # frames — carried over from earlier motion-energy findings
SAVGOL_POLYORDER = 2
N_PER_VIDEO_SAMPLES = 50  # scope 1
N_CATEGORY_SAMPLES = 10  # scope 2
LOAD_BATCH = 100  # videos per batch inside a per-category/chunk unit
CHUNK_SIZE = 500  # videos per resumable unit in the global scope

TYPE_ORDER = ["pose", "left_hand", "right_hand", "face"]
TYPE_COLORS = {
    "pose": "tab:blue",
    "left_hand": "tab:orange",
    "right_hand": "tab:green",
    "face": "tab:red",
}

print("DATA_DIR:", DATA_DIR)
print("cache   :", MOTION_CACHE_DIR.resolve())


## 1. Reusable core (TODO §1.1)

Metadata loading, then two measurement functions (`sb.recognize.interp.motion_energy`,
already unit-tested against a real npz), then one plotting function per signal kind —
built once, used by all three scopes. Each gets an immediate smoke test on one seeded
random video.


In [ ]:
# ============================================================
# train.csv + test.csv -> one metadata table for every scope below
# ============================================================
meta = pd.concat([
    pd.read_csv(DATA_DIR / "train.csv").assign(split="train"),
    pd.read_csv(DATA_DIR / "test.csv").assign(split="test"),
], ignore_index=True).sort_values("npz_relpath").reset_index(drop=True)

print(f"{len(meta):,} videos · {meta['sign'].nunique()} signs · "
      f"{meta['participant_id'].nunique()} participants "
      f"({(meta['split'] == 'train').sum():,} train / {(meta['split'] == 'test').sum():,} test)")
meta.head(3)


In [ ]:
# ============================================================
# load_positions() — thin wrapper over load_npz, smoke-tested here
# ============================================================
_smoke_row = meta.sample(1, random_state=SEED).iloc[0]
_smoke_path = DATA_DIR / _smoke_row["npz_relpath"]
_smoke_xyz = me.load_positions(_smoke_path, coords="xyz")
_smoke_xy = _smoke_xyz[:, :, :2]  # drop z FIRST, before any motion-energy computation

print(_smoke_row["npz_relpath"], "-", _smoke_row["sign"])
print(f"frames={_smoke_xyz.shape[0]}  landmarks={_smoke_xyz.shape[1]}  "
      f"xyz nan-frac={np.isnan(_smoke_xyz).mean():.3f}")


In [ ]:
# ============================================================
# compute_motion_unit() — the one function every scope calls
# ============================================================
def compute_motion_unit(
    npz_path: Path, video_id: str, sign: str,
    window: int = SAVGOL_WINDOW, polyorder: int = SAVGOL_POLYORDER,
) -> dict[str, pd.DataFrame]:
    '''One video's npz -> {"landmarks": df, "angles": df}, both tidy long format.

    landmarks: video_id, sign, type, landmark_index, rms_speed (xy), n_valid_transitions
    angles:    video_id, sign, angle_name, rms_angular_speed, angle_mean, angle_std
    '''
    raw_xyz = me.load_positions(npz_path, coords="xyz")
    raw_xy = raw_xyz[:, :, :2]  # drop z first (TODO §1.8 / §7.7; report §2.3)

    lm = me.landmark_motion_energy(raw_xy, window, polyorder)
    lm.insert(0, "sign", sign)
    lm.insert(0, "video_id", video_id)

    ang = me.joint_angle_motion(raw_xyz, window, polyorder)
    ang.insert(0, "sign", sign)
    ang.insert(0, "video_id", video_id)

    return {"landmarks": lm, "angles": ang}


# smoke test — reuse the loaded single video
_smoke_units = compute_motion_unit(_smoke_path, str(_smoke_row["sequence_id"]), _smoke_row["sign"])
assert len(_smoke_units["landmarks"]) == 543, "expected 543 landmark rows"
assert len(_smoke_units["angles"]) == len(ANGLE_NAMES), f"expected {len(ANGLE_NAMES)} angle rows"
print(_smoke_units["landmarks"].groupby("type", observed=True)["rms_speed"].describe().round(4))
_smoke_units["angles"].sort_values("rms_angular_speed", ascending=False).head(5)


In [ ]:
# ============================================================
# plot_motion_gridspec() / plot_angle_bars() — one layout per signal kind,
# shared by all three scopes
# ============================================================
def plot_motion_gridspec(df: pd.DataFrame, title: str,
                         save_path: Path | None = None, show: bool = True) -> None:
    '''Landmark RMS speed (xy) — combined overview on top (landmarks ordered
    pose -> left_hand -> right_hand -> face), one panel per type below.

    `df` must hold ONE row per (type, landmark_index). Accepts either a
    per-video frame (`rms_speed`) or an aggregate one (`rms_speed_mean` +
    `rms_speed_std`), detected automatically — std shows as error bars in the
    per-type panels. NaN landmarks (never observed) render as gaps.
    '''
    val_col = "rms_speed_mean" if "rms_speed_mean" in df.columns else "rms_speed"
    std_col = "rms_speed_std" if "rms_speed_std" in df.columns else None

    fig = plt.figure(figsize=(16, 9))
    gs = fig.add_gridspec(2, 4, height_ratios=[1.3, 1], hspace=0.35, wspace=0.28)

    ax_top = fig.add_subplot(gs[0, :])
    offset = 0
    for t in TYPE_ORDER:
        sub = df[df["type"] == t].sort_values("landmark_index")
        xs = np.arange(len(sub)) + offset
        ax_top.bar(xs, sub[val_col].to_numpy(), width=1.0,
                   color=TYPE_COLORS[t], label=t)
        offset += len(sub) + 10
    ax_top.set_title("all landmarks — ordered pose → left_hand → right_hand → face (xy)")
    ax_top.set_ylabel("RMS speed (xy)")
    ax_top.set_xticks([])
    ax_top.legend(ncols=4, fontsize=9)

    for i, t in enumerate(TYPE_ORDER):
        ax = fig.add_subplot(gs[1, i])
        sub = df[df["type"] == t].sort_values("landmark_index")
        yerr = sub[std_col].fillna(0).to_numpy() if std_col else None
        ax.bar(sub["landmark_index"], sub[val_col].to_numpy(), width=1.0,
               color=TYPE_COLORS[t], yerr=yerr,
               error_kw={"elinewidth": 0.4, "alpha": 0.5})
        ax.set_title(t, fontsize=10)
        ax.set_xlabel("landmark_index", fontsize=8)
        if i == 0:
            ax.set_ylabel("RMS speed (xy)")

    fig.suptitle(title, fontsize=13, y=0.99)
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=110, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)


def plot_angle_bars(df: pd.DataFrame, title: str,
                    save_path: Path | None = None, show: bool = True) -> None:
    '''RMS angular speed ("change of angles") — one bar per joint angle,
    sorted descending. Accepts per-video (`rms_angular_speed`) or aggregate
    (`rms_angular_speed_mean` + `_std`) frames.'''
    val_col = "rms_angular_speed_mean" if "rms_angular_speed_mean" in df.columns else "rms_angular_speed"
    std_col = "rms_angular_speed_std" if "rms_angular_speed_std" in df.columns else None

    sub = df.sort_values(val_col, ascending=False)
    fig, ax = plt.subplots(figsize=(14, 5))
    yerr = sub[std_col].fillna(0).to_numpy() if std_col else None
    colors = ["tab:purple" if n.startswith("L_") else "tab:brown" if n.startswith("R_") else "gray"
              for n in sub["angle_name"]]
    ax.bar(np.arange(len(sub)), sub[val_col].to_numpy(), color=colors,
           yerr=yerr, error_kw={"elinewidth": 0.4, "alpha": 0.5})
    ax.set_xticks(np.arange(len(sub)))
    ax.set_xticklabels(sub["angle_name"], rotation=75, fontsize=7, ha="right")
    ax.set_ylabel("RMS angular speed (deg/frame)")
    ax.set_title(title, fontsize=12)
    fig.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=110, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)


# smoke test
plot_motion_gridspec(
    _smoke_units["landmarks"],
    title=f"video {_smoke_row['sequence_id']} — '{_smoke_row['sign']}'")
plot_angle_bars(
    _smoke_units["angles"],
    title=f"video {_smoke_row['sequence_id']} — '{_smoke_row['sign']}' — joint angle change")


## 2. Resumable caching / state management (TODO §1.2)

One manifest per scope at `data/cache/gislr/motion_analysis/<scope>_manifest.json`,
tracking `{unit_id: {status, timestamp, artifacts: {landmarks, angles}[, error]}}`.
The invariants (unchanged from the pre-npz notebook, `artifacts` now plural):

1. both per-unit parquets are written **before** the unit is marked `done`;
2. manifest saves are atomic (temp file + `os.replace`);
3. on re-run, `done` units with both artifacts present are skipped, everything else
   (including `failed`) is retried;
4. final aggregation reads only cached per-unit files — never raw npz.


In [ ]:
# ============================================================
# Manifest helpers + the resumable driver every scope runs through
# ============================================================
def _manifest_path(scope: str) -> Path:
    return MOTION_CACHE_DIR / f"{scope}_manifest.json"


def _unit_path(scope: str, kind: str, unit_id: str) -> Path:
    return MOTION_CACHE_DIR / scope / kind / f"{unit_id}.parquet"


def load_manifest(scope: str) -> dict:
    p = _manifest_path(scope)
    return json.loads(p.read_text()) if p.exists() else {}


def save_manifest(scope: str, manifest: dict) -> None:
    '''Atomic write — a crash mid-save can't corrupt the manifest.'''
    p = _manifest_path(scope)
    tmp = p.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(manifest, indent=2))
    os.replace(tmp, p)


def process_units(scope: str, unit_ids: list, process_fn, desc: str | None = None) -> dict:
    '''Run `process_fn(unit_id) -> {"landmarks": DataFrame, "angles": DataFrame}`
    over units, resumably.

    Skips units already `done` with both artifacts present, retries `failed`
    ones, records exceptions in the manifest instead of aborting the run.
    '''
    for kind in ("landmarks", "angles"):
        (MOTION_CACHE_DIR / scope / kind).mkdir(parents=True, exist_ok=True)
    manifest = load_manifest(scope)
    n_skip = n_ok = n_fail = 0
    for uid in tqdm([str(u) for u in unit_ids], desc=desc or scope):
        entry = manifest.get(uid)
        if (entry and entry["status"] == "done"
                and all(Path(p).exists() for p in entry["artifacts"].values())):
            n_skip += 1
            continue
        try:
            out = process_fn(uid)
            artifacts = {}
            for kind, df in out.items():
                artifact = _unit_path(scope, kind, uid)
                df.to_parquet(artifact, index=False)  # artifacts FIRST …
                artifacts[kind] = str(artifact)
            manifest[uid] = {"status": "done",  # … then mark done
                             "timestamp": datetime.now(timezone.utc).isoformat(),
                             "artifacts": artifacts}
            n_ok += 1
        except Exception as e:
            manifest[uid] = {"status": "failed",
                             "timestamp": datetime.now(timezone.utc).isoformat(),
                             "artifacts": {}, "error": repr(e)}
            n_fail += 1
        save_manifest(scope, manifest)
    print(f"[{scope}] done={n_ok}  skipped={n_skip}  failed={n_fail}")
    if n_fail:
        print("failed units:",
              {k: v["error"] for k, v in manifest.items() if v["status"] == "failed"})
    return manifest


def load_cached_units(scope: str, kind: str, unit_ids: list) -> pd.DataFrame:
    '''Aggregation input: reads ONLY cached per-unit parquets, never raw npz.'''
    frames = [pd.read_parquet(_unit_path(scope, kind, str(u)))
              for u in unit_ids if _unit_path(scope, kind, str(u)).exists()]
    return pd.concat(frames, ignore_index=True)


## 3. Scope 1 — per-video, 50 random samples (TODO §1.3)

Unit = one video: load → compute (landmarks + angles) → **save both PNGs** → cache
both parquets. The exact sample (seed + paths) is recorded to
`per_video_sample.json` for reproducibility.


In [ ]:
SCOPE1 = "per_video"
PLOTS_DIR = MOTION_CACHE_DIR / SCOPE1 / "plots"

rng = np.random.default_rng(SEED)
_idx = np.sort(rng.choice(len(meta), size=N_PER_VIDEO_SAMPLES, replace=False))
scope1_meta = meta.iloc[_idx][["npz_relpath", "sequence_id", "sign"]].reset_index(drop=True)

(MOTION_CACHE_DIR / f"{SCOPE1}_sample.json").write_text(json.dumps(
    {"seed": SEED, "n": N_PER_VIDEO_SAMPLES, "npz_relpaths": scope1_meta["npz_relpath"].tolist()},
    indent=2))

_row_by_vid = {str(r["sequence_id"]): r for _, r in scope1_meta.iterrows()}


def _process_one_video(video_id: str) -> dict[str, pd.DataFrame]:
    row = _row_by_vid[video_id]
    out = compute_motion_unit(DATA_DIR / row["npz_relpath"], video_id, row["sign"])
    plot_motion_gridspec(out["landmarks"], title=f"video {video_id} — '{row['sign']}'",
                         save_path=PLOTS_DIR / f"{video_id}_landmarks.png", show=False)
    plot_angle_bars(out["angles"], title=f"video {video_id} — '{row['sign']}' — angle change",
                    save_path=PLOTS_DIR / f"{video_id}_angles.png", show=False)
    return out


manifest1 = process_units(SCOPE1, scope1_meta["sequence_id"].astype(str).tolist(), _process_one_video)

scope1_landmarks = load_cached_units(SCOPE1, "landmarks", scope1_meta["sequence_id"].astype(str).tolist())
scope1_angles = load_cached_units(SCOPE1, "angles", scope1_meta["sequence_id"].astype(str).tolist())
scope1_landmarks.to_parquet(MOTION_CACHE_DIR / SCOPE1 / "landmarks" / "summary.parquet", index=False)
scope1_angles.to_parquet(MOTION_CACHE_DIR / SCOPE1 / "angles" / "summary.parquet", index=False)
print(f"landmarks: {scope1_landmarks.shape[0]:,} rows "
      f"({scope1_landmarks['video_id'].nunique()} videos × 543 landmarks)")
print(f"angles   : {scope1_angles.shape[0]:,} rows "
      f"({scope1_angles['video_id'].nunique()} videos × {len(ANGLE_NAMES)} angles)")
scope1_landmarks.groupby("type", observed=True)["rms_speed"].describe().round(4)


In [ ]:
# two example per-video figure pairs inline (all 50 are in per_video/plots/)
for _vid in scope1_meta["sequence_id"].astype(str).head(2):
    _sign = _row_by_vid[_vid]["sign"]
    plot_motion_gridspec(pd.read_parquet(_unit_path(SCOPE1, "landmarks", _vid)),
                         title=f"video {_vid} — '{_sign}'")
    plot_angle_bars(pd.read_parquet(_unit_path(SCOPE1, "angles", _vid)),
                    title=f"video {_vid} — '{_sign}' — angle change")


## 4. Scope 2 — per-category, 10 sampled signs (TODO §1.4)

Unit = one sign: batched load of **all** its videos → per-video RMS (landmarks +
angles) → aggregate mean + std per landmark / per angle. The std is what a future
within-class consistency analysis would feed on. `n_videos` counts videos where the
landmark had at least one valid transition (NaN rms excluded).


In [ ]:
SCOPE2 = "per_category"

rng = np.random.default_rng(SEED)
scope2_signs = sorted(rng.choice(np.sort(meta["sign"].unique()),
                                 size=N_CATEGORY_SAMPLES, replace=False).tolist())

(MOTION_CACHE_DIR / f"{SCOPE2}_sample.json").write_text(json.dumps(
    {"seed": SEED, "n": N_CATEGORY_SAMPLES, "signs": scope2_signs}, indent=2))
print("signs:", scope2_signs)
print("videos per sign:",
      meta[meta["sign"].isin(scope2_signs)].groupby("sign").size().to_dict())


def _process_one_sign(sign: str) -> dict[str, pd.DataFrame]:
    rows = meta.loc[meta["sign"] == sign]
    per_video_lm, per_video_ang = [], []
    for _, row in rows.iterrows():
        out = compute_motion_unit(DATA_DIR / row["npz_relpath"], str(row["sequence_id"]), sign)
        per_video_lm.append(out["landmarks"])
        per_video_ang.append(out["angles"])
    lm = pd.concat(per_video_lm, ignore_index=True)
    ang = pd.concat(per_video_ang, ignore_index=True)

    lm_agg = (lm.groupby(["type", "landmark_index"], observed=True)["rms_speed"]
                .agg(rms_speed_mean="mean", rms_speed_std="std", n_videos="count")
                .reset_index())
    lm_agg.insert(0, "sign", sign)

    ang_agg = (ang.groupby("angle_name", observed=True)["rms_angular_speed"]
                 .agg(rms_angular_speed_mean="mean", rms_angular_speed_std="std", n_videos="count")
                 .reset_index())
    ang_agg.insert(0, "sign", sign)

    return {"landmarks": lm_agg, "angles": ang_agg}


manifest2 = process_units(SCOPE2, scope2_signs, _process_one_sign)

scope2_landmarks = load_cached_units(SCOPE2, "landmarks", scope2_signs)
scope2_angles = load_cached_units(SCOPE2, "angles", scope2_signs)
scope2_landmarks.to_parquet(MOTION_CACHE_DIR / SCOPE2 / "landmarks" / "summary.parquet", index=False)
scope2_angles.to_parquet(MOTION_CACHE_DIR / SCOPE2 / "angles" / "summary.parquet", index=False)

_ex_lm = scope2_landmarks[scope2_landmarks["sign"] == scope2_signs[0]]
_ex_ang = scope2_angles[scope2_angles["sign"] == scope2_signs[0]]
plot_motion_gridspec(
    _ex_lm, title=f"sign '{scope2_signs[0]}' — mean ± std over {int(_ex_lm['n_videos'].max())} videos")
plot_angle_bars(
    _ex_ang, title=f"sign '{scope2_signs[0]}' — angle change, mean ± std over {int(_ex_ang['n_videos'].max())} videos")


## 5. Scope 3 — global, entire dataset (TODO §1.5)

**Decision:** chunks of 500 videos through the same load → compute → cache path as
scope 2, manifest tracking **chunk** completion. Chunk membership is deterministic
(paths sorted), so resuming after an interrupt always rebuilds the same chunks. No
DuckDB here (unlike the pre-npz version): there's no cross-file JOIN to push into
SQL — each npz is self-contained and metadata is one CSV read — so the final
aggregation is a plain streaming pandas reduction over the cached chunk parquets
(sum / sum-of-squares / count per group, memory-bounded the same way the DuckDB
`GROUP BY` was).

Each chunk artifact stores the *per-video* RMS rows (landmarks + angles). Safe to
interrupt; re-run to resume.


In [ ]:
SCOPE3 = "global"

_all_rows = meta.sort_values("npz_relpath").reset_index(drop=True)  # deterministic chunking
_chunks = [_all_rows.iloc[i:i + CHUNK_SIZE] for i in range(0, len(_all_rows), CHUNK_SIZE)]
chunk_ids = [f"chunk_{i:05d}" for i in range(len(_chunks))]
_chunk_by_id = dict(zip(chunk_ids, _chunks))
print(f"{len(_all_rows):,} videos → {len(_chunks)} chunks of ≤{CHUNK_SIZE}")


def _process_one_chunk(chunk_id: str) -> dict[str, pd.DataFrame]:
    rows = _chunk_by_id[chunk_id]
    per_video_lm, per_video_ang = [], []
    for _, row in rows.iterrows():
        out = compute_motion_unit(DATA_DIR / row["npz_relpath"], str(row["sequence_id"]), row["sign"])
        per_video_lm.append(out["landmarks"])
        per_video_ang.append(out["angles"])
    return {
        "landmarks": pd.concat(per_video_lm, ignore_index=True),
        "angles": pd.concat(per_video_ang, ignore_index=True),
    }


manifest3 = process_units(SCOPE3, chunk_ids, _process_one_chunk)


In [ ]:
# ============================================================
# Global aggregation — streaming sum/sumsq/count over the cached chunk
# parquets only (memory-bounded: one chunk's rows in memory at a time)
# ============================================================
def _streaming_aggregate(scope: str, kind: str, chunk_ids: list, group_cols: list, value_col: str) -> pd.DataFrame:
    acc = None
    for cid in chunk_ids:
        p = _unit_path(scope, kind, cid)
        if not p.exists():
            continue
        df = pd.read_parquet(p, columns=[*group_cols, value_col])
        df = df[df[value_col].notna()]
        g = df.groupby(group_cols, observed=True)[value_col]
        part = g.agg(sum="sum", sumsq=lambda s: float((s ** 2).sum()), n="count")
        acc = part if acc is None else acc.add(part, fill_value=0.0)
    mean = acc["sum"] / acc["n"]
    var = (acc["sumsq"] / acc["n"] - mean ** 2).clip(lower=0.0)
    out = pd.DataFrame({f"{value_col}_mean": mean, f"{value_col}_std": np.sqrt(var),
                        "n_videos": acc["n"].astype(int)}).reset_index()
    return out


_n_done = sum(1 for v in manifest3.values() if v["status"] == "done")
if _n_done < len(chunk_ids):
    print(f"global scope incomplete: {_n_done}/{len(chunk_ids)} chunks done — "
          f"re-run the previous cell to resume; aggregation skipped.")
else:
    global_landmarks = _streaming_aggregate(SCOPE3, "landmarks", chunk_ids,
                                            ["type", "landmark_index"], "rms_speed")
    global_angles = _streaming_aggregate(SCOPE3, "angles", chunk_ids,
                                         ["angle_name"], "rms_angular_speed")
    global_landmarks.to_parquet(MOTION_CACHE_DIR / SCOPE3 / "landmarks" / "summary.parquet", index=False)
    global_angles.to_parquet(MOTION_CACHE_DIR / SCOPE3 / "angles" / "summary.parquet", index=False)

    plot_motion_gridspec(
        global_landmarks,
        title=f"GLOBAL — mean ± std across {len(_all_rows):,} videos (xy)",
        save_path=MOTION_CACHE_DIR / SCOPE3 / "global_overview_landmarks.png")
    plot_angle_bars(
        global_angles,
        title=f"GLOBAL — angle change, mean ± std across {len(_all_rows):,} videos",
        save_path=MOTION_CACHE_DIR / SCOPE3 / "global_overview_angles.png")
    display(global_landmarks.groupby("type", observed=True)["rms_speed_mean"].describe().round(4))


## 6. Cross-scope comparison (TODO §1.6)

Do the small samples (50 videos / 10 signs) reproduce the global per-landmark and
per-angle motion pattern? If they do (high rank correlation), sample-based analyses
can be trusted for downstream landmark-importance decisions; if not, they can't.


In [ ]:
_landmark_summaries = {
    "per_video_sample": MOTION_CACHE_DIR / "per_video" / "landmarks" / "summary.parquet",
    "per_category_sample": MOTION_CACHE_DIR / "per_category" / "landmarks" / "summary.parquet",
    "global": MOTION_CACHE_DIR / "global" / "landmarks" / "summary.parquet",
}
_angle_summaries = {
    "per_video_sample": MOTION_CACHE_DIR / "per_video" / "angles" / "summary.parquet",
    "per_category_sample": MOTION_CACHE_DIR / "per_category" / "angles" / "summary.parquet",
    "global": MOTION_CACHE_DIR / "global" / "angles" / "summary.parquet",
}
_missing = [k for k, p in {**_landmark_summaries, **_angle_summaries}.items() if not p.exists()]
if _missing:
    print(f"missing summaries — run the corresponding scope(s) first.")
else:
    s1 = (pd.read_parquet(_landmark_summaries["per_video_sample"])
          .groupby(["type", "landmark_index"], observed=True)["rms_speed"]
          .mean().rename("per_video_sample"))
    s2 = (pd.read_parquet(_landmark_summaries["per_category_sample"])
          .groupby(["type", "landmark_index"], observed=True)["rms_speed_mean"]
          .mean().rename("per_category_sample"))
    sg = (pd.read_parquet(_landmark_summaries["global"])
          .set_index(["type", "landmark_index"])["rms_speed_mean"].rename("global"))
    cross = pd.concat([s1, s2, sg], axis=1).reset_index()

    a1 = pd.read_parquet(_angle_summaries["per_video_sample"]).groupby("angle_name")["rms_angular_speed"].mean().rename("per_video_sample")
    a2 = pd.read_parquet(_angle_summaries["per_category_sample"]).groupby("angle_name")["rms_angular_speed_mean"].mean().rename("per_category_sample")
    ag = pd.read_parquet(_angle_summaries["global"]).set_index("angle_name")["rms_angular_speed_mean"].rename("global")
    cross_angles = pd.concat([a1, a2, ag], axis=1).reset_index()

    print("-- landmarks (xy) --")
    for c in ["per_video_sample", "per_category_sample"]:
        m = cross[[c, "global"]].dropna()
        rho, _ = spearmanr(m[c], m["global"])
        print(f"{c:>22} vs global: spearman rho = {rho:.3f}  (n={len(m)} landmarks)")

    print("-- joint angles (change of angle) --")
    for c in ["per_video_sample", "per_category_sample"]:
        m = cross_angles[[c, "global"]].dropna()
        rho, _ = spearmanr(m[c], m["global"])
        print(f"{c:>22} vs global: spearman rho = {rho:.3f}  (n={len(m)} angles)")

    # overlay: landmarks ordered pose -> left_hand -> right_hand -> face
    fig, ax = plt.subplots(figsize=(16, 5))
    ordered = pd.concat(
        [cross[cross["type"] == t].sort_values("landmark_index") for t in TYPE_ORDER],
        ignore_index=True)
    x = np.arange(len(ordered))
    for c, style in [("global", dict(color="black", lw=1.6)),
                     ("per_category_sample", dict(color="tab:green", lw=1.0, alpha=0.8)),
                     ("per_video_sample", dict(color="tab:purple", lw=1.0, alpha=0.8))]:
        ax.plot(x, ordered[c], label=c, **style)
    b = 0
    for t in TYPE_ORDER:
        n = (ordered["type"] == t).sum()
        ax.axvspan(b, b + n, color=TYPE_COLORS[t], alpha=0.06)
        ax.text(b + n / 2, ax.get_ylim()[1] * 0.95, t, ha="center", fontsize=9)
        b += n
    ax.set_title("Cross-scope comparison — mean RMS speed per landmark (xy)")
    ax.set_ylabel("RMS speed (xy)")
    ax.set_xlabel("landmarks (pose → left_hand → right_hand → face)")
    ax.set_xticks([])
    ax.legend()
    fig.savefig(MOTION_CACHE_DIR / "cross_scope_landmarks.png", dpi=110, bbox_inches="tight")
    plt.show()

    # overlay: joint angles
    fig, ax = plt.subplots(figsize=(16, 5))
    ordered_a = cross_angles.set_index("angle_name").loc[ANGLE_NAMES].reset_index()
    xa = np.arange(len(ordered_a))
    for c, style in [("global", dict(color="black", lw=1.6)),
                     ("per_category_sample", dict(color="tab:green", lw=1.0, alpha=0.8)),
                     ("per_video_sample", dict(color="tab:purple", lw=1.0, alpha=0.8))]:
        ax.plot(xa, ordered_a[c], label=c, **style)
    ax.set_title("Cross-scope comparison — mean RMS angular speed per joint angle")
    ax.set_ylabel("RMS angular speed (deg/frame)")
    ax.set_xticks(xa)
    ax.set_xticklabels(ordered_a["angle_name"], rotation=75, fontsize=7, ha="right")
    ax.legend()
    fig.tight_layout()
    fig.savefig(MOTION_CACHE_DIR / "cross_scope_angles.png", dpi=110, bbox_inches="tight")
    plt.show()


## Caveats & what this notebook deliberately does NOT do (TODO §1.6–1.7)

> **Findings here are not conclusive** until cross-checked against the
> discriminability instrument (`gislr.0.dataset.feature-discriminability.ipynb`) and
> the competition-suggested landmark subset. High motion ≠ high discriminative
> value — a landmark can move a lot identically across all signs, and the previous
> run of this analysis found discriminability ~uncorrelated with motion energy
> (rho −0.12, `docs/reports/subset-comparison.md`).

Out of scope (separate, already-existing instruments):
- within-class / cross-class discriminability — `gislr.0.dataset.feature-discriminability.ipynb`
  and `gislr.0.dataset.subset-comparison.ipynb` (the latter still needs its own
  npz rebuild, tracked separately in TODO §0.1);
- gradient saliency / SHAP (needs a trained model — this is pre-training analysis);
- spectrogram-format conversion (TODO §5).

**Follow-up once this notebook has been run:** update
`docs/reports/motion-energy.md` with the new xy-native global numbers and the new
joint-angle-motion findings — the current report describes the pre-npz run and its
own xy-vs-xyz sample-only decomposition (report §2.3), which this notebook now
supersedes at full-dataset scale.
